In [58]:
import pandas as pd
import numpy as np
import cmdstanpy
from cmdstanpy import CmdStanModel

### DATA PREPROCESSING

In [59]:
# --- CONFIGURATION & MODEL DEPENDENCIES ---
STAN_FILE = "rl_ddm_bias.stan"  
TRANSITION_PROB_COMMON = 0.7 
#TEST CONFIGURATION
TEST_SUBJECT_COUNT = 151
TEST_TRIAL_COUNT = 200

df = pd.read_csv("final_dataset.csv")

In [60]:
import json
import numpy as np
import os
import pandas as pd

# Assuming S is defined earlier (S = len(test_subjects))
# Find S again to ensure the JSON file has the correct size
unique_subjects = df['subject_id'].unique()
S = len(unique_subjects[:TEST_SUBJECT_COUNT]) # e.g., 5

def create_safe_init_file(S, filename="rl_ddm_inits.json"):
    safe_init_ter = 0.05   
    safe_init_a = 1.0      
    safe_init_z = 0.5      
    safe_init_v_coeff = 1.0
    safe_init_alpha = 0.5
    safe_init_lambda = 0.5
    safe_init_omega = 0.5
    safe_init_beta_s1 = 5.0
    safe_init_pers = 0.0

    init_data = {
        'alpha': [safe_init_alpha] * S,
        'lambda': [safe_init_lambda] * S,
        'omega': [safe_init_omega] * S,
        'beta_s1': [safe_init_beta_s1] * S,
        'pers': [safe_init_pers] * S,
        'v_coeff': [safe_init_v_coeff] * S,
        'a': [safe_init_a] * S,
        'ter': [safe_init_ter] * S, 
        'z': [safe_init_z] * S
    }

    file_path = os.path.join(os.getcwd(), filename)
    with open(file_path, 'w') as f:
        json.dump(init_data, f)
    
    print(f"✅ Created safe initialization file: {file_path}")
    return file_path

INIT_FILE_PATH = create_safe_init_file(S)

✅ Created safe initialization file: /Users/oguz/Desktop/thesis-project/data/rl_ddm_inits.json


In [61]:
from cmdstanpy import CmdStanModel
import pandas as pd
import numpy as np

# Filter for stability and Stan compatibility
df = df[
    (df['rt_2'] >= 150) &    # Keep trials with valid RT (used for DDM)
    (df['choice_1'] != 0) &  
    (df['choice_2'] != 0) &  
    (df['state'] != 0)       
]
# =============================================================================
# 2. SLICE DATA FOR SMALL TEST
# =============================================================================
unique_subjects = df['subject_id'].unique()
test_subjects = unique_subjects[:TEST_SUBJECT_COUNT] 

S = len(test_subjects)
T_max = TEST_TRIAL_COUNT 

# Initialize matrices based on the test T_max
c1_mat = np.zeros((S, T_max), dtype=int)
s2_mat = np.zeros((S, T_max), dtype=int)  # <-- Renamed matrix initialization
c2_mat = np.zeros((S, T_max), dtype=int)
r_mat = np.zeros((S, T_max), dtype=float)
rt2_mat = np.zeros((S, T_max), dtype=float) # <-- NEW: RT Matrix
T_per_subject = []
prior_choice_vec = np.zeros(S, dtype=int)

# 3. FILL MATRICES WITH SLICED DATA
for i, subj in enumerate(test_subjects):
    subj_data = df[df['subject_id'] == subj]
    
    # Limit to TEST_TRIAL_COUNT
    subj_data = subj_data.head(TEST_TRIAL_COUNT) 
    
    n_trials = len(subj_data)
    T_per_subject.append(n_trials)
    
    # Fill the matrices row by row (up to n_trials, the rest remains 0-padded)
    c1_mat[i, :n_trials]    = subj_data['choice_1'].values.astype(int)
    s2_mat[i, :n_trials]    = subj_data['state'].values.astype(int) 
    c2_mat[i, :n_trials]    = subj_data['choice_2'].values.astype(int)
    r_mat[i, :n_trials]     = subj_data['reward'].values.astype(float)
    rt2_mat[i, :n_trials]   = subj_data['rt_2'].values.astype(float) # <-- NEW: Populate RT
    
    # Get the very first choice for stickiness prior initialization
    prior_choice_vec[i] = subj_data['choice_1'].iloc[0]

# =============================================================================
# 4. FINAL STAN DATA DICTIONARY (with list conversion fix)
# =============================================================================

stan_data = {
    'S': S,
    'T_max': T_max,
    'T': T_per_subject, 
    'c1': c1_mat.tolist(),
    's2': s2_mat.tolist(),        
    'c2': c2_mat.tolist(),
    'r': r_mat.tolist(),
    'rt2': rt2_mat.tolist(),     
    't_common': TRANSITION_PROB_COMMON, 
}

# The DDM Stan file does NOT use 'prior_choice' because the initial Q-matrix
# is set to 0.5. The stickiness 'prev_c1' starts at 0 within the model loop.
# But since your DDM model uses it, let's include it for consistency:
stan_data['prior_choice'] = prior_choice_vec.tolist() 

print(f"✅ Small test data dictionary created for {S} subjects x {T_max} max trials.")

# =============================================================================
# 5. RUNNING THE MODEL
# NOTE: Using slightly higher 'inits' than 0 to prevent DDM crash.
model = CmdStanModel(stan_file=STAN_FILE)

fit = model.sample(
    data=stan_data,
    chains=4,
    iter_warmup=1000,    
    iter_sampling=1000,
    adapt_delta=0.95,   
    max_treedepth=12,
    inits=INIT_FILE_PATH,       
    show_progress=True,
    show_console=True
    )

summary = fit.summary()
print(summary.loc[[c for c in summary.index if 'alpha' in c or 'omega' in c or 'a' in c or 'ter' in c]].head(8))

22:09:59 - cmdstanpy - INFO - Chain [1] start processing
22:09:59 - cmdstanpy - INFO - Chain [2] start processing
22:09:59 - cmdstanpy - INFO - Chain [3] start processing
22:09:59 - cmdstanpy - INFO - Chain [4] start processing


✅ Small test data dictionary created for 151 subjects x 200 max trials.
Chain [1] method = sample (Default)
Chain [1] sample
Chain [1] num_samples = 1000 (Default)
Chain [1] num_warmup = 1000 (Default)
Chain [1] save_warmup = false (Default)
Chain [1] thin = 1 (Default)
Chain [1] adapt
Chain [1] engaged = true (Default)
Chain [1] gamma = 0.05 (Default)
Chain [1] delta = 0.95
Chain [1] kappa = 0.75 (Default)
Chain [1] t0 = 10 (Default)
Chain [1] init_buffer = 75 (Default)
Chain [1] term_buffer = 50 (Default)
Chain [1] window = 25 (Default)
Chain [1] save_metric = false (Default)
Chain [1] algorithm = hmc (Default)
Chain [1] hmc
Chain [1] engine = nuts (Default)
Chain [1] nuts
Chain [1] max_depth = 12
Chain [1] metric = diag_e (Default)
Chain [1] metric_file =  (Default)
Chain [1] stepsize = 1 (Default)
Chain [1] stepsize_jitter = 0 (Default)
Chain [1] num_chains = 1 (Default)
Chain [1] id = 1 (Default)
Chain [1] data
Chain [1] file = /var/folders/rb/hnd2mx114hnblrd6d4t0p02w0000gn/T/tmp_

01:19:23 - cmdstanpy - INFO - Chain [4] done processing


Chain [4] 
Chain [4] Elapsed Time: 8092.19 seconds (Warm-up)
Chain [4] 3274.14 seconds (Sampling)
Chain [4] 11366.3 seconds (Total)
Chain [4] 
Chain [4] 
Chain [3] Iteration: 2000 / 2000 [100%]  (Sampling)


01:20:35 - cmdstanpy - INFO - Chain [3] done processing


Chain [3] 
Chain [3] Elapsed Time: 8173.28 seconds (Warm-up)
Chain [3] 3265.18 seconds (Sampling)
Chain [3] 11438.5 seconds (Total)
Chain [3] 
Chain [3] 
Chain [2] Iteration: 2000 / 2000 [100%]  (Sampling)


01:21:15 - cmdstanpy - INFO - Chain [2] done processing


Chain [2] 
Chain [2] Elapsed Time: 8207.83 seconds (Warm-up)
Chain [2] 3270.9 seconds (Sampling)
Chain [2] 11478.7 seconds (Total)
Chain [2] 
Chain [2] 
Chain [1] Iteration: 1800 / 2000 [ 90%]  (Sampling)
Chain [1] Iteration: 1900 / 2000 [ 95%]  (Sampling)
Chain [1] Iteration: 2000 / 2000 [100%]  (Sampling)


01:29:04 - cmdstanpy - INFO - Chain [1] done processing
01:29:04 - cmdstanpy - WARNING - Non-fatal error during sampling:
Exception: Exception: wiener_lpdf: Random variable  = 1.45762, but must be greater than nondecision time = 1.39168e+15 (in 'rl_ddm_bias.stan', line 6, column 6 to column 62) (in 'rl_ddm_bias.stan', line 125, column 6 to column 81)
	Exception: Exception: wiener_lpdf: Random variable  = 1.45762, but must be greater than nondecision time = 7.18965e+14 (in 'rl_ddm_bias.stan', line 6, column 6 to column 62) (in 'rl_ddm_bias.stan', line 125, column 6 to column 81)
	Exception: Exception: wiener_lpdf: Random variable  = 1.45762, but must be greater than nondecision time = 433.721 (in 'rl_ddm_bias.stan', line 6, column 6 to column 62) (in 'rl_ddm_bias.stan', line 125, column 6 to column 81)
	Exception: Exception: wiener_lpdf: Random variable  = 0.48359, but must be greater than nondecision time = 0.585951 (in 'rl_ddm_bias.stan', line 6, column 6 to column 62) (in 'rl_ddm_bia

Chain [1] 
Chain [1] Elapsed Time: 9066.25 seconds (Warm-up)
Chain [1] 2881.04 seconds (Sampling)
Chain [1] 11947.3 seconds (Total)
Chain [1] 
Chain [1] 
              Mean      MCSE    StdDev       MAD        5%       50%  \
alpha[1]  0.347043  0.000680  0.048908  0.049282  0.269497  0.345919   
alpha[2]  0.643780  0.001723  0.123335  0.125709  0.444427  0.642669   
alpha[3]  0.315175  0.001966  0.101915  0.086626  0.178254  0.300482   
alpha[4]  0.791927  0.001333  0.088583  0.092041  0.641362  0.795433   
alpha[5]  0.187593  0.002326  0.143008  0.105503  0.032088  0.149442   
alpha[6]  0.601438  0.002126  0.150804  0.153722  0.353740  0.599613   
alpha[7]  0.654282  0.000781  0.060281  0.060347  0.556893  0.653269   
alpha[8]  0.481271  0.001064  0.073441  0.072945  0.363771  0.479020   

               95%  ESS_bulk  ESS_tail  ESS_bulk/s     R_hat  
alpha[1]  0.430089   5281.15   2920.67    0.416125  0.999927  
alpha[2]  0.850665   4994.71   2323.27    0.393555  1.000300  
alpha[3]

In [62]:
import numpy as np
import pandas as pd

# --- CORRECTION ---
# 1. Use the correct variable names from the DDM Stan file output.
# These variables contain 1.0 for a correct prediction, 0.0 for an incorrect prediction.
accuracy_s1_all = fit.stan_variable('accuracy_s1')
accuracy_s2_all = fit.stan_variable('accuracy_s2')

# N_draws is calculated from the shape of the resulting array
N_draws = accuracy_s1_all.shape[0]

# --- Setup Masking (Necessary to exclude padded trials) ---
S = stan_data['S']
T_max = stan_data['T_max']
T_per_subject = stan_data['T']

mask = np.zeros((S, T_max), dtype=bool)
for s in range(S):
    # Only mask up to the number of actual trials (T[s])
    mask[s, :T_per_subject[s]] = True
mask_tiled = np.tile(mask, (N_draws, 1, 1))

# --- Calculate Mean Accuracy ---

# 1. Apply the mask to the Stage 1 accuracy results
stage1_accuracy_valid = accuracy_s1_all[mask_tiled]
# 2. Calculate the mean (which is the proportion of 1s/correct predictions)
mean_accuracy_stage1 = np.mean(stage1_accuracy_valid)

# 3. Apply the mask to the Stage 2 accuracy results
stage2_accuracy_valid = accuracy_s2_all[mask_tiled]
mean_accuracy_stage2 = np.mean(stage2_accuracy_valid)

print(f"\n--- Posterior Predictive Accuracy (Across All Subjects & Trials) ---")
print(f"Stage 1 (c1) Predictive Accuracy: {mean_accuracy_stage1:.4f} ({mean_accuracy_stage1*100:.2f}%)")
print(f"Stage 2 (c2) Predictive Accuracy: {mean_accuracy_stage2:.4f} ({mean_accuracy_stage2*100:.2f}%)")


--- Posterior Predictive Accuracy (Across All Subjects & Trials) ---
Stage 1 (c1) Predictive Accuracy: 0.7597 (75.97%)
Stage 2 (c2) Predictive Accuracy: 0.7114 (71.14%)


In [63]:
# Assuming you just calculated mean_accuracy_stage1 and mean_accuracy_stage2

rmse_stage1 = np.sqrt(1 - mean_accuracy_stage1)
rmse_stage2 = np.sqrt(1 - mean_accuracy_stage2)

print(f"\n--- Root Mean Square Error (RMSE) ---")
print(f"Stage 1 (c1) RMSE: {rmse_stage1:.4f}")
print(f"Stage 2 (c2) RMSE: {rmse_stage2:.4f}")


--- Root Mean Square Error (RMSE) ---
Stage 1 (c1) RMSE: 0.4902
Stage 2 (c2) RMSE: 0.5372


In [64]:
import pandas as pd
import numpy as np

# Assuming 'fit' is the CmdStanPy fit object from your last run.

# 1. Get the full summary table
full_summary_df = fit.summary()

# Define the base names of the core RL/DDM parameters for filtering.
# This excludes trial-level diagnostics like accuracy_s1 and v_t.
rl_ddm_parameter_bases = [
    'alpha', 'lambda', 'omega', 'beta_s1', 'pers', 
    'v_coeff', 'a', 'ter', 'z', 'log_lik_trial' # Log_lik_trial is included for LOO/WAIC.
]

# 2. Create a boolean mask to filter the DataFrame index
mask = pd.Series([False] * len(full_summary_df), index=full_summary_df.index)

for base in rl_ddm_parameter_bases:
    # Use OR (|) logic to combine masks for all core parameters
    mask = mask | full_summary_df.index.str.startswith(base)

# 3. Filter the DataFrame to keep only the parameter estimates and log-likelihoods
filtered_rl_ddm_summary_df = full_summary_df[mask]

print("\n--- Filtered RL-DDM Summary DataFrame Created ---")
print(f"Shape of the filtered summary (Parameters x Stats): {filtered_rl_ddm_summary_df.shape}")
print("Head of the Filtered Summary (Used for R-hat Check):")
print(filtered_rl_ddm_summary_df.head(10).to_markdown(floatfmt=".4f"))

# --- Now you can run your R-hat calculation code block ---
# Select the R_hat column from the filtered summary DataFrame
rhat_values = filtered_rl_ddm_summary_df['R_hat']

# Find the key summary statistics for R-hat
max_rhat = rhat_values.max()
n_rhat_above_1_01 = (rhat_values > 1.01).sum()
mean_rhat = rhat_values.mean()

print("\n--- Overall R-hat Diagnostics for RL-DDM ---")
print(f"Total number of parameters checked (S x Parameters x Trials): {len(rhat_values)}")
print(f"Maximum R-hat found: {max_rhat:.5f}")
print(f"Mean R-hat across all parameters: {mean_rhat:.5f}") 
print(f"Number of parameters with R-hat > 1.01: {n_rhat_above_1_01}")


--- Filtered RL-DDM Summary DataFrame Created ---
Shape of the filtered summary (Parameters x Stats): (91959, 11)
Head of the Filtered Summary (Used for R-hat Check):
|           |   Mean |   MCSE |   StdDev |    MAD |     5% |    50% |    95% |   ESS_bulk |   ESS_tail |   ESS_bulk/s |   R_hat |
|:----------|-------:|-------:|---------:|-------:|-------:|-------:|-------:|-----------:|-----------:|-------------:|--------:|
| alpha[1]  | 0.3470 | 0.0007 |   0.0489 | 0.0493 | 0.2695 | 0.3459 | 0.4301 |  5281.1500 |  2920.6700 |       0.4161 |  0.9999 |
| alpha[2]  | 0.6438 | 0.0017 |   0.1233 | 0.1257 | 0.4444 | 0.6427 | 0.8507 |  4994.7100 |  2323.2700 |       0.3936 |  1.0003 |
| alpha[3]  | 0.3152 | 0.0020 |   0.1019 | 0.0866 | 0.1783 | 0.3005 | 0.5034 |  3333.8800 |  2323.4300 |       0.2627 |  1.0014 |
| alpha[4]  | 0.7919 | 0.0013 |   0.0886 | 0.0920 | 0.6414 | 0.7954 | 0.9367 |  4148.0200 |  2241.8500 |       0.3268 |  0.9999 |
| alpha[5]  | 0.1876 | 0.0023 |   0.1430 | 0.1055 | 

In [65]:
import pandas as pd
import numpy as np

# Assuming 'fit' is the CmdStanPy fit object from your last run.
# Assuming 'full_summary_df' is the result of fit.summary().

# --- If full_summary_df is not yet available, run this line: ---
# full_summary_df = fit.summary()
# -----------------------------------------------------------------


# Define the base names of the RL DDM parameters (excluding trial-level outputs)
rl_ddm_parameter_bases = [
    'alpha', 'lambda', 'omega', 'beta_s1', 'pers', 
    'v_coeff', 'a', 'ter', 'z', 'log_lik_trial'
]

# Create a mask that selects rows starting with one of the parameter base names
mask = pd.Series([False] * len(full_summary_df), index=full_summary_df.index)

for base in rl_ddm_parameter_bases:
    # Use OR (|) logic to combine masks for all RL DDM parameters
    mask = mask | full_summary_df.index.str.startswith(base)

# Filter the DataFrame to keep only the parameter estimates
rl_ddm_parameters_df = full_summary_df[mask]

# Display a quick look at the estimated parameters
print("\n--- RL DDM Parameter Estimates (Mean & Diagnostics) ---")
print(rl_ddm_parameters_df.head(10))

# Save the final table of parameter estimates to CSV
rl_ddm_parameters_df.to_csv("final_rl_ddm_parameter_estimates.csv")
print("\n✅ Successfully saved RL DDM parameter estimates to final_rl_ddm_parameter_estimates.csv")



--- RL DDM Parameter Estimates (Mean & Diagnostics) ---
               Mean      MCSE    StdDev       MAD        5%       50%  \
alpha[1]   0.347043  0.000680  0.048908  0.049282  0.269497  0.345919   
alpha[2]   0.643780  0.001723  0.123335  0.125709  0.444427  0.642669   
alpha[3]   0.315175  0.001966  0.101915  0.086626  0.178254  0.300482   
alpha[4]   0.791927  0.001333  0.088583  0.092041  0.641362  0.795433   
alpha[5]   0.187593  0.002326  0.143008  0.105503  0.032088  0.149442   
alpha[6]   0.601438  0.002126  0.150804  0.153722  0.353740  0.599613   
alpha[7]   0.654282  0.000781  0.060281  0.060347  0.556893  0.653269   
alpha[8]   0.481271  0.001064  0.073441  0.072945  0.363771  0.479020   
alpha[9]   0.176761  0.003173  0.202811  0.125764  0.002356  0.093002   
alpha[10]  0.500225  0.001252  0.089922  0.090552  0.359534  0.496538   

                95%  ESS_bulk  ESS_tail  ESS_bulk/s     R_hat  
alpha[1]   0.430089   5281.15   2920.67    0.416125  0.999927  
alpha[2]   

In [66]:
import pandas as pd
import numpy as np

# --- Configuration ---
TARGET_SUBJECT_INDEX = 0 
TARGET_SUBJECT_ID = test_subjects[TARGET_SUBJECT_INDEX]
T_common = TRANSITION_PROB_COMMON
# ---------------------

# --- 1. Extract Subject Data and Mean Parameter Estimates ---
# Use the main 'df' to access the subject's full data
subj_data_obs = df[df['subject_id'] == TARGET_SUBJECT_ID].head(T_max).reset_index(drop=True)

# Dictionary to hold the mean estimates for Subject 1 (Index 0)
subj_params = {}
param_bases = ['alpha', 'lambda', 'omega', 'beta_s1', 'pers', 
               'v_coeff', 'a', 'ter', 'z']

for base in param_bases:
    param_name = f'{base}[{TARGET_SUBJECT_INDEX + 1}]'
    try:
        # Assuming filtered_rl_ddm_summary_df is available from filtering the fit.summary()
        mean_value = filtered_rl_ddm_summary_df.loc[param_name]['Mean']
        subj_params[base] = mean_value
    except KeyError as e:
        print(f"Error: Parameter {param_name} not found. Please ensure the model ran correctly and the summary was filtered.")
        raise e
        
print(f"\n--- Running Prediction Simulation for Subject ID: {TARGET_SUBJECT_ID} ---")

# --- 2. Run Trial-by-Trial Simulation and Comparison ---
Q = np.full((3, 2), 0.5) # Stan indices [1, 2, 3] -> NumPy indices [0, 1, 2]
prev_c1 = 0
results = []

for t in range(len(subj_data_obs)):
    trial = subj_data_obs.iloc[t]
    
    # Skip invalid trials
    if trial['choice_2'] == 0:
        continue

    # --- RL Value Calculation (Hybrid) ---
    max_s2 = np.max(Q[1, :]) # Stage 2 Q-values (NumPy index 1)
    max_s3 = np.max(Q[2, :]) # Stage 3 Q-values (NumPy index 2)
    
    q_mb_1 = T_common * max_s2 + (1 - T_common) * max_s3
    q_mb_2 = (1 - T_common) * max_s2 + T_common * max_s3
    
    q_mf_1 = Q[0, 0] # Stage 1 Q-values (NumPy index 0)
    q_mf_2 = Q[0, 1] 
    
    q_net_1 = subj_params['omega'] * q_mb_1 + (1 - subj_params['omega']) * q_mf_1
    q_net_2 = subj_params['omega'] * q_mb_2 + (1 - subj_params['omega']) * q_mf_2

    # Stickiness
    stick = 0.0
    if prev_c1 == 1: stick = subj_params['pers']
    elif prev_c1 == 2: stick = -subj_params['pers']

    # --- Stage 1 Prediction (The Q2-Q1 Reversal Fix) ---
    # Logit difference must be Q2 - Q1 for the model to correctly predict C1 vs C2
    logit_s1 = subj_params['beta_s1'] * (q_net_2 - q_net_1) + stick 
    
    # Prediction: If logit is POSITIVE (Q2 > Q1), it predicts Choice 2.
    pred_c1 = 2 if logit_s1 >= 0 else 1
    
    # --- Stage 2 Prediction (DDM) ---
    act1 = trial['choice_1']
    act2 = trial['choice_2']
    reward = trial['reward']
    state_idx = trial['state'] - 1 # Convert Stan State index (2 or 3) to NumPy index (1 or 2)
    
    # Q-difference driving the DDM drift
    delta_Q = Q[state_idx, 0] - Q[state_idx, 1] # Q(A1) - Q(A2)
    drift_t = subj_params['v_coeff'] * delta_Q
    
    # Prediction: Positive drift (Q(A1) > Q(A2)) predicts Choice 1
    pred_c2 = 1 if drift_t >= 0 else 2 
    
    # --- Q-Value Update ---
    act2_np = act2 - 1 
    act1_np = act1 - 1
    
    # Update Stage 2
    pe2 = reward - Q[state_idx, act2_np] 
    Q[state_idx, act2_np] += subj_params['alpha'] * pe2
    
    # Update Stage 1
    if act1 != 0:
        val_next = Q[state_idx, act2_np]
        pe1 = val_next - Q[0, act1_np]
        Q[0, act1_np] += subj_params['alpha'] * pe1 + subj_params['lambda'] * subj_params['alpha'] * pe2
        prev_c1 = act1
    else:
        prev_c1 = 0

    # --- Record Results ---
    match_c1 = '✅' if pred_c1 == act1 else '❌'
    match_c2 = '✅' if pred_c2 == act2 else '❌'
    results.append({
        'Trial': t + 1,
        'Actual_C1': act1,
        'Pred_C1': pred_c1,
        'Match_C1': match_c1,
        'Actual_C2': act2,
        'Pred_C2': pred_c2,
        'Match_C2': match_c2,
        'Q_diff_S1': q_net_1 - q_net_2,
        'Q_diff_S2': delta_Q,
        'Drift': drift_t,
    })

# Print Results
results_df = pd.DataFrame(results)
# Use to_string as a robust fallback for printing large tables
print(results_df.to_string(index=False))


--- Running Prediction Simulation for Subject ID: sub1 ---
 Trial  Actual_C1  Pred_C1 Match_C1  Actual_C2  Pred_C2 Match_C2  Q_diff_S1  Q_diff_S2     Drift
     1          1        2        ❌          1        1        ✅   0.000000   0.000000  0.000000
     2          1        1        ✅          1        1        ✅   0.090671   0.173521  0.806014
     3          2        1        ❌          1        2        ❌  -0.011434  -0.060219 -0.279722
     4          2        1        ❌          2        1        ❌   0.029889   0.000000  0.000000
     5          2        2        ✅          2        2        ✅  -0.075123  -0.212842 -0.988661
     6          1        2        ❌          2        2        ✅  -0.060631  -0.173521 -0.806014
     7          1        2        ❌          1        2        ❌  -0.044241  -0.386364 -1.794675
     8          2        2        ✅          1        1        ✅  -0.072127   0.060219  0.279722
     9          1        2        ❌          2        2        ✅   

In [67]:
import numpy as np
import pandas as pd

# --- 1. Load Accuracy and Setup Masking ---

# Load the trial-by-trial accuracy (1.0 for match, 0.0 for mismatch)
# Dimensions: (N_draws, S, T_max)
accuracy_s1_all = fit.stan_variable('accuracy_s1')
accuracy_s2_all = fit.stan_variable('accuracy_s2')

N_draws = accuracy_s1_all.shape[0]
S = stan_data['S']
T_per_subject = stan_data['T']
T_max = stan_data['T_max']

# Create the mask for valid (non-padded) trials
mask = np.zeros((S, T_max), dtype=bool)
for s in range(S):
    mask[s, :T_per_subject[s]] = True

# --- 2. Calculate Mean Accuracy Per Subject ---

# Step A: Average across all posterior draws (draws dimension)
# Resulting shape: (S, T_max)
avg_accuracy_s1_per_trial = np.mean(accuracy_s1_all, axis=0)
avg_accuracy_s2_per_trial = np.mean(accuracy_s2_all, axis=0)

# Step B: Average across trials (T_max dimension), respecting the mask

subject_results = []
for s in range(S):
    subject_id = test_subjects[s]
    n_trials = T_per_subject[s]
    
    # Apply the mask to get only the valid trials for the current subject
    valid_s1_accuracies = avg_accuracy_s1_per_trial[s, :n_trials]
    valid_s2_accuracies = avg_accuracy_s2_per_trial[s, :n_trials]
    
    # Calculate the mean (overall predictive accuracy) for the subject
    pred_c1 = np.mean(valid_s1_accuracies)
    pred_c2 = np.mean(valid_s2_accuracies)
    
    subject_results.append({
        'subject_id': subject_id,
        'pred_c1_accuracy': pred_c1 * 100,
        'pred_c2_accuracy': pred_c2 * 100,
        'n_trials': n_trials
    })

# --- 3. Display Results ---

accuracy_df = pd.DataFrame(subject_results)

print("\n--- Individual Subject Predictive Accuracy (%) ---")
print(accuracy_df.to_markdown(floatfmt=".2f", index=False))

# You can also save this table for your thesis:
# accuracy_df.to_csv("subject_predictive_accuracy.csv", index=False)


--- Individual Subject Predictive Accuracy (%) ---
| subject_id   |   pred_c1_accuracy |   pred_c2_accuracy |   n_trials |
|:-------------|-------------------:|-------------------:|-----------:|
| sub1         |              67.63 |              91.03 |        199 |
| sub10        |              48.09 |              70.38 |        200 |
| sub100       |              79.72 |              63.06 |        133 |
| sub101       |              94.30 |              72.36 |        199 |
| sub102       |              62.81 |              57.63 |        148 |
| sub103       |              86.31 |              49.82 |        177 |
| sub104       |              90.29 |              85.50 |        200 |
| sub105       |              95.50 |              81.90 |        200 |
| sub106       |              68.78 |              40.82 |        157 |
| sub107       |              66.48 |              73.09 |        199 |
| sub108       |              60.12 |              67.86 |        196 |
| sub109    

In [69]:
import numpy as np
import pandas as pd

# --- Configuration (Necessary variables defined in environment) ---
S = stan_data['S'] 
T_per_subject = stan_data['T']
T_max = stan_data['T_max']
test_subjects = np.array(test_subjects) # Ensure subject IDs are accessible
# -----------------------------------------------------------------

# Load the trial-by-trial accuracy (Dimensions: N_draws x S x T_max)
accuracy_s1_all = fit.stan_variable('accuracy_s1')
accuracy_s2_all = fit.stan_variable('accuracy_s2')

N_draws = accuracy_s1_all.shape[0]

# --- 2. Calculate Mean Accuracy Per Subject ---

# Step A: Average across all posterior draws (draws dimension)
# Resulting shape: (S, T_max)
avg_accuracy_s1_per_trial = np.mean(accuracy_s1_all, axis=0)
avg_accuracy_s2_per_trial = np.mean(accuracy_s2_all, axis=0)

subject_results = []
for s in range(S):
    subject_id = test_subjects[s]
    n_trials = T_per_subject[s]
    
    # Apply the mask to get only the valid trials (0 to n_trials)
    valid_s1_accuracies = avg_accuracy_s1_per_trial[s, :n_trials]
    valid_s2_accuracies = avg_accuracy_s2_per_trial[s, :n_trials]
    
    # Calculate the mean (overall predictive accuracy) for the subject
    pred_c1 = np.mean(valid_s1_accuracies)
    pred_c2 = np.mean(valid_s2_accuracies)
    
    subject_results.append({
        'subject_id': subject_id,
        'C1_Accuracy': pred_c1 * 100,
        'C2_Accuracy': pred_c2 * 100,
        'N_valid_trials': n_trials
    })

# --- 3. Final Output ---
accuracy_df = pd.DataFrame(subject_results)

print("\n--- Final Predictive Accuracy Per Subject (Direct from Fit) ---")
print(accuracy_df.to_markdown(floatfmt=".2f", index=False))

overall_c1_mean = accuracy_df['C1_Accuracy'].mean()
overall_c2_mean = accuracy_df['C2_Accuracy'].mean()

print("\n--- Overall Model Performance ---")
print(f"Overall Mean C1 Accuracy: {overall_c1_mean:.2f}%")
print(f"Overall Mean C2 Accuracy: {overall_c2_mean:.2f}%")

# Save the final summary table
accuracy_df.to_csv("all_subjects_predictive_accuracy_summary_fit_rl_ddm.csv", index=False)


--- Final Predictive Accuracy Per Subject (Direct from Fit) ---
| subject_id   |   C1_Accuracy |   C2_Accuracy |   N_valid_trials |
|:-------------|--------------:|--------------:|-----------------:|
| sub1         |         67.63 |         91.03 |              199 |
| sub10        |         48.09 |         70.38 |              200 |
| sub100       |         79.72 |         63.06 |              133 |
| sub101       |         94.30 |         72.36 |              199 |
| sub102       |         62.81 |         57.63 |              148 |
| sub103       |         86.31 |         49.82 |              177 |
| sub104       |         90.29 |         85.50 |              200 |
| sub105       |         95.50 |         81.90 |              200 |
| sub106       |         68.78 |         40.82 |              157 |
| sub107       |         66.48 |         73.09 |              199 |
| sub108       |         60.12 |         67.86 |              196 |
| sub109       |         76.15 |         74.65 |   

In [70]:
import numpy as np

# --- ESS Diagnostic Code ---

# Select the ESS columns from the filtered summary DataFrame
ess_bulk_values = full_summary_df['ESS_bulk']
ess_tail_values = full_summary_df['ESS_tail']

# --- ESS_bulk (Reliability of Mean/Median) ---
min_ess_bulk = ess_bulk_values.min()
max_ess_bulk = ess_bulk_values.max()
mean_ess_bulk = ess_bulk_values.mean()

# --- ESS_tail (Reliability of Credible Intervals) ---
min_ess_tail = ess_tail_values.min()
max_ess_tail = ess_tail_values.max()
mean_ess_tail = ess_tail_values.mean()

print("\n--- Overall ESS Diagnostics (Across All Core RL Parameters) ---")
print(f"Total number of core parameters checked: {len(ess_bulk_values)}")


print("\n### ESS_bulk (Main Body of Posterior) ###")
print(f"Minimum ESS_bulk: {min_ess_bulk:.1f}")
print(f"Maximum ESS_bulk: {max_ess_bulk:.1f}")
print(f"Mean ESS_bulk: {mean_ess_bulk:.1f}")

print("\n### ESS_tail (Credible Intervals) ###")
print(f"Minimum ESS_tail: {min_ess_tail:.1f}")
print(f"Maximum ESS_tail: {max_ess_tail:.1f}")
print(f"Mean ESS_tail: {mean_ess_tail:.1f}")



--- Overall ESS Diagnostics (Across All Core RL Parameters) ---
Total number of core parameters checked: 122160

### ESS_bulk (Main Body of Posterior) ###
Minimum ESS_bulk: 13.9
Maximum ESS_bulk: 14065.5
Mean ESS_bulk: 5000.0

### ESS_tail (Credible Intervals) ###
Minimum ESS_tail: 13.9
Maximum ESS_tail: 7748.5
Mean ESS_tail: 3025.8
